

---

# Surface material detection using Full Scene imagery

In [ ]:
PROMPT = """Act as a civil engineering material analyst specializing in road surfaces. Your task is to analyze the provided Google Street View image and accurately identify the dominant road surface material present
Your analysis must follow these steps to ensure accuracy:
1.  **Identify Road Surface Area**: Clearly delineate the primary road surface in the image, ignoring sidewalks, shoulders, or surrounding terrain.
2.  **Visual Evidence Extraction**: For the identified road surface, describe the visual cues that indicate its material type. Focus specifically on:
    *   **Texture & Micro-structure**: Look for characteristics such as:
        *   **Paved**: Smooth, uniform, often dark or light grey. May show aggregate, cracks, patches, or lane markings. If asphalt, a granular, somewhat coarse texture. If concrete, a finer, often broom-finished texture with expansion joints.
        *   **Gravel**: Loose, irregularly shaped stones of varying sizes with visible gaps and an uneven surface. Often exhibits tire tracks or displacement.
        *   **Mud**: Soft, wet, or dried soil, often with ruts, puddles, or deep impressions from vehicles. Can vary widely in color and consistency.
        *   **Dirt**: Unpaved, dry, compacted soil. Less uniform than paved, but more stable than mud, often exhibiting dust or tire marks.
    *   **Reflectance & Specularity**: Observe how light interacts with the surface. Is it matte (dirt, some mud), somewhat reflective (wet mud, new asphalt), or does it show clear highlights (wet paved roads, standing water)?
    *   **Contextual Cues**: Consider environmental factors such as surrounding vegetation, drainage, and road infrastructure (e.g., presence of road signs, guardrails nearby, but not directly on the road surface).
**Output Format**:
Provide your findings in a structured JSON format:
{
  "road_surface_material": "[Material classification: Paved (Asphalt/Concrete), Gravel, Mud, Dirt, or Other]",
  "confidence_score": "[0-100%]",
  "visual_reasoning": "[1-3 sentences describing specific visual evidence supporting the classification, e.g., 'Surface exhibits uniform dark gray color with visible aggregate and clear lane markings, consistent with asphalt pavement.']",
  "image_url": "{image_url}"
}
**Note:**
**Important Considerations:**
*   Focus exclusively on the material directly comprising the main driving surface.
*   Ignore temporary conditions like standing water or debris unless they are definitive indicators of the underlying road material.
*   If the material is ambiguous or mixed, classify based on the dominant type and note ambiguity in reasoning."""

In [ ]:
limit_count = 10 #@param {type:"integer"}
import pandas_gbq
import vertexai
from vertexai.preview.generative_models import GenerativeModel, Part
import urllib.parse

# Query the BigQuery table
project_id = "YOUR_PROJECT_ID"

# Auto-detect GCP project if placeholder is unchanged
if project_id == "YOUR_PROJECT_ID":
    try:
        import google.auth
        _, auth_project = google.auth.default()
        if auth_project:
            project_id = auth_project
            print(f"[INFO] Auto-detected Google Cloud Project: {project_id}")
    except Exception:
        pass #@param {type:"string"}
track_id = 'o1:MlvyRlxbxkIcf0PSQk2cMA_4:5001ee' #@param {type:"string"}
dataset_id = 'imagery_insights___us' #@param {type:"string"}
table_name = 'pano_observations_latest' #@param {type:"string"}

sql_query = f"""
SELECT
  gcs_uri
FROM
  `{project_id}`.`{dataset_id}`.`{table_name}`
WHERE
  capture_id = '{track_id}' OR pano_id = '{track_id}' OR observation_id = '{track_id}'
LIMIT {limit_count}
"""
df = pandas_gbq.read_gbq(sql_query, project_id, dialect="standard")

if not df.empty:
    gcs_uri_for_vertex_ai = df['gcs_uri'].iloc[0]
    print(f"Final GCS URI for Vertex AI: {gcs_uri_for_vertex_ai}")

    try:
        vertexai.init(project=project_id, location="global")
        model_name = "gemini-3.5-flash" #@param {type:"string"}
        model = GenerativeModel(model_name)

        prompt = PROMPT 
        image_part = Part.from_uri(uri=gcs_uri_for_vertex_ai, mime_type="image/jpeg")

        response = model.generate_content([image_part, prompt])
        print("Gemini Model Response:")
        print(response.text)

    except Exception as e:
        print(f"An unexpected error occurred: {e}")
else:
    print("No observation found for the given ID.")